In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from datetime import datetime
from urllib.parse import urlparse
from google.colab import files

# Install rich untuk tampilan log keren
import subprocess
subprocess.run(["pip", "install", "rich", "-q"])

from rich.console import Console
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich import box
from rich.live import Live
from rich.columns import Columns
from rich.rule import Rule

console = Console()

# =========================
# INPUT
# =========================
console.print(Panel.fit(
    "[bold cyan]🕷  SITEMAP CRAWLER[/bold cyan]\n"
    "[dim]Universal XML Sitemap Extractor[/dim]",
    border_style="cyan"
))

SITEMAP_URL = console.input("\n[bold yellow]📎 Masukkan URL Sitemap:[/bold yellow] ").strip()

def get_domain_name(url: str) -> str:
    try:
        parsed = urlparse(url)
        domain = parsed.netloc or parsed.path
        domain = re.sub(r'^www\.', '', domain)
        safe   = re.sub(r'[^\w]', '_', domain)
        return safe
    except Exception:
        return "sitemap"

DOMAIN_NAME = get_domain_name(SITEMAP_URL)
OUTPUT_FILE = f"{DOMAIN_NAME}_sitemap_full.csv"
START_TIME  = datetime.now()

console.print(f"\n  [green]✔[/green] Domain   : [bold]{DOMAIN_NAME}[/bold]")
console.print(f"  [green]✔[/green] Output   : [bold cyan]{OUTPUT_FILE}[/bold cyan]")
console.print(f"  [green]✔[/green] Mulai    : [dim]{START_TIME.strftime('%Y-%m-%d %H:%M:%S')}[/dim]\n")

# =========================
# CONFIG
# =========================
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; SitemapCrawler/3.0; +https://example.com/bot)"
}

MAX_SITEMAPS    = 500
REQUEST_TIMEOUT = 30
DELAY_BETWEEN   = 0.5

YEAR_PATTERN = r'\b(19|20)\d{2}\b'

FILE_EXT_MAP = {
    "image"   : {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp",
                 ".tiff", ".svg", ".ico", ".avif"},
    "video"   : {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"},
    "audio"   : {".mp3", ".wav", ".aac", ".ogg", ".m4a", ".flac"},
    "document": {".pdf", ".doc", ".docx", ".xls", ".xlsx",
                 ".ppt", ".pptx", ".txt", ".rtf"},
    "archive" : {".zip", ".rar", ".7z", ".tar", ".gz"},
    "feed"    : {".xml", ".rss", ".atom"},
}

ENTRY_TYPE_STYLE = {
    "page_url"    : "[white]",
    "image_url"   : "[cyan]",
    "video_url"   : "[magenta]",
    "news_url"    : "[yellow]",
    "image_file"  : "[blue]",
    "video_file"  : "[bright_magenta]",
    "audio_file"  : "[green]",
    "document_file": "[bright_yellow]",
    "feed_file"   : "[bright_cyan]",
    "file"        : "[dim]",
}

# =========================
# HELPERS
# =========================
def safe_text(tag):
    return tag.get_text(strip=True) if tag else None

def get_ext_from_url(u: str):
    try:
        path = urlparse(u).path.lower()
        if "." in path:
            ext = "." + path.split(".")[-1]
            return ext if len(ext) <= 6 else None
        return None
    except Exception:
        return None

def classify_by_ext(u: str):
    ext = get_ext_from_url(u)
    if not ext:
        return None, None
    for kind, exts in FILE_EXT_MAP.items():
        if ext in exts:
            return kind, ext
    return "file", ext

def is_valid_url(loc: str):
    return bool(loc and loc.startswith(("http://", "https://")))

def request_xml(url, timeout=REQUEST_TIMEOUT):
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    return resp.content

def parse_xml(xml_bytes):
    soup = BeautifulSoup(xml_bytes, "xml")
    root = soup.find(["sitemapindex", "urlset"])
    return soup, root

def short_url(url, max_len=65):
    return url if len(url) <= max_len else url[:max_len - 3] + "..."

# =========================
# EXTRACTOR: urlset
# =========================
def extract_urlset(soup, sitemap_source):
    rows = []
    for url_tag in soup.find_all("url"):
        loc        = safe_text(url_tag.find("loc"))
        lastmod    = safe_text(url_tag.find("lastmod"))
        changefreq = safe_text(url_tag.find("changefreq"))
        priority   = safe_text(url_tag.find("priority"))

        contains_year = bool(re.search(YEAR_PATTERN, loc or ""))

        # Image
        image_locs, image_titles, image_captions = [], [], []
        for img in url_tag.find_all(re.compile(r"^(image:image|image)$", re.I)):
            il = safe_text(img.find(re.compile(r"^(image:loc|loc)$",         re.I)))
            it = safe_text(img.find(re.compile(r"^(image:title|title)$",     re.I)))
            ic = safe_text(img.find(re.compile(r"^(image:caption|caption)$", re.I)))
            if il: image_locs.append(il)
            if it: image_titles.append(it)
            if ic: image_captions.append(ic)

        # Video
        video_titles, video_thumbnails, video_durations = [], [], []
        video_content_locs, video_player_locs           = [], []
        for vid in url_tag.find_all(re.compile(r"^(video:video|video)$", re.I)):
            vt  = safe_text(vid.find(re.compile(r"^(video:title|title)$",                 re.I)))
            vth = safe_text(vid.find(re.compile(r"^(video:thumbnail_loc|thumbnail_loc)$", re.I)))
            vd  = safe_text(vid.find(re.compile(r"^(video:duration|duration)$",           re.I)))
            vc  = safe_text(vid.find(re.compile(r"^(video:content_loc|content_loc)$",     re.I)))
            vp  = safe_text(vid.find(re.compile(r"^(video:player_loc|player_loc)$",       re.I)))
            if vt:  video_titles.append(vt)
            if vth: video_thumbnails.append(vth)
            if vd:  video_durations.append(vd)
            if vc:  video_content_locs.append(vc)
            if vp:  video_player_locs.append(vp)

        # News
        news_publication = news_pub_date = news_title = news_language = None
        news_tag = url_tag.find(re.compile(r"^(news:news|news)$", re.I))
        if news_tag:
            pub = news_tag.find(re.compile(r"^(news:publication|publication)$", re.I))
            if pub:
                news_publication = safe_text(pub.find(re.compile(r"^(news:name|name)$",         re.I)))
                news_language    = safe_text(pub.find(re.compile(r"^(news:language|language)$", re.I)))
            news_pub_date = safe_text(news_tag.find(re.compile(r"^(news:publication_date|publication_date)$", re.I)))
            news_title    = safe_text(news_tag.find(re.compile(r"^(news:title|title)$",          re.I)))

        kind_by_ext, ext = classify_by_ext(loc) if loc else (None, None)
        has_image = len(image_locs) > 0
        has_video = len(video_titles) > 0 or len(video_content_locs) > 0 or len(video_player_locs) > 0
        has_news  = news_tag is not None

        if has_news:       entry_type = "news_url"
        elif has_video:    entry_type = "video_url"
        elif has_image:    entry_type = "image_url"
        elif kind_by_ext:  entry_type = f"{kind_by_ext}_file"
        else:              entry_type = "page_url"

        rows.append({
            "sitemap_source"       : sitemap_source,
            "loc"                  : loc,
            "lastmod"              : lastmod,
            "changefreq"           : changefreq,
            "priority"             : priority,
            "contains_year"        : contains_year,
            "entry_type"           : entry_type,
            "url_valid"            : is_valid_url(loc),
            "file_ext"             : ext,
            "image_count"          : len(image_locs),
            "image_locs"           : " | ".join(image_locs)     if image_locs     else None,
            "image_titles"         : " | ".join(image_titles)   if image_titles   else None,
            "image_captions"       : " | ".join(image_captions) if image_captions else None,
            "video_count"          : max(len(video_titles), len(video_content_locs), len(video_player_locs)),
            "video_titles"         : " | ".join(video_titles)       if video_titles       else None,
            "video_thumbnails"     : " | ".join(video_thumbnails)   if video_thumbnails   else None,
            "video_durations"      : " | ".join(video_durations)    if video_durations    else None,
            "video_content_locs"   : " | ".join(video_content_locs) if video_content_locs else None,
            "video_player_locs"    : " | ".join(video_player_locs)  if video_player_locs  else None,
            "news_publication"     : news_publication,
            "news_language"        : news_language,
            "news_publication_date": news_pub_date,
            "news_title"           : news_title,
        })

    return rows

# =========================
# EXTRACTOR: sitemapindex
# =========================
def extract_sitemapindex(soup):
    children = []
    for sm in soup.find_all("sitemap"):
        loc     = safe_text(sm.find("loc"))
        lastmod = safe_text(sm.find("lastmod"))
        if loc:
            children.append({"loc": loc, "lastmod": lastmod})
    return children

# =========================
# MAIN CRAWLER
# =========================
def crawl_sitemap(start_url, max_sitemaps=MAX_SITEMAPS):
    visited       = set()
    queue         = [start_url]
    all_rows      = []
    summary_index = []
    total_fetched = 0
    error_count   = 0

    console.rule("[bold cyan]🚀 MULAI CRAWLING[/bold cyan]")

    with Progress(
        SpinnerColumn(spinner_name="dots", style="cyan"),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(bar_width=30, style="cyan", complete_style="green"),
        TextColumn("[bold green]{task.fields[urls]}[/bold green] URL"),
        TextColumn("·"),
        TextColumn("[dim]{task.fields[sitemap]}[/dim]"),
        TimeElapsedColumn(),
        console=console,
        refresh_per_second=10,
    ) as progress:

        task = progress.add_task(
            "[cyan]Crawling...",
            total=None,
            urls=0,
            sitemap="starting...",
        )

        while queue:
            current = queue.pop(0)
            if current in visited:
                continue
            visited.add(current)

            if total_fetched >= max_sitemaps:
                console.print(f"\n[yellow]⚠  Batas max_sitemaps={max_sitemaps} tercapai.[/yellow]")
                break

            progress.update(
                task,
                description=f"[cyan]Fetching [{total_fetched + 1}]",
                sitemap=short_url(current, 55),
            )

            try:
                xml_bytes  = request_xml(current)
                soup, root = parse_xml(xml_bytes)
                total_fetched += 1

                if not root:
                    console.print(f"  [yellow]⚠[/yellow]  Tidak ada root tag: [dim]{short_url(current)}[/dim]")
                    continue

                root_name = root.name.lower()

                if root_name == "sitemapindex":
                    children = extract_sitemapindex(soup)
                    console.print(
                        f"  [bold blue]📂 INDEX[/bold blue]   "
                        f"[dim]{short_url(current, 58)}[/dim]  "
                        f"→  [bold]{len(children)}[/bold] child sitemaps"
                    )
                    for ch in children:
                        summary_index.append({
                            "sitemap_url" : ch["loc"],
                            "sitemap_type": "child",
                            "lastmod"     : ch["lastmod"],
                            "url_count"   : None,
                        })
                        if ch["loc"] not in visited:
                            queue.append(ch["loc"])

                elif root_name == "urlset":
                    rows = extract_urlset(soup, sitemap_source=current)
                    all_rows.extend(rows)

                    type_counts = {}
                    for r in rows:
                        type_counts[r["entry_type"]] = type_counts.get(r["entry_type"], 0) + 1
                    breakdown = "  ".join(
                        f"{ENTRY_TYPE_STYLE.get(k,'[white]')}{v} {k}[/]"
                        for k, v in sorted(type_counts.items(), key=lambda x: -x[1])
                    )

                    console.print(
                        f"  [bold green]📄 URLSET[/bold green]  "
                        f"[dim]{short_url(current, 58)}[/dim]  "
                        f"→  [bold green]{len(rows)}[/bold green] URLs  {breakdown}"
                    )

                    for s in summary_index:
                        if s["sitemap_url"] == current:
                            s["url_count"] = len(rows)
                            break
                    else:
                        summary_index.append({
                            "sitemap_url" : current,
                            "sitemap_type": "root_urlset",
                            "lastmod"     : None,
                            "url_count"   : len(rows),
                        })

                    progress.update(task, urls=len(all_rows))

                time.sleep(DELAY_BETWEEN)

            except requests.HTTPError as e:
                error_count += 1
                console.print(f"  [red]✗ HTTP {e.response.status_code}[/red]  [dim]{short_url(current)}[/dim]")
            except requests.Timeout:
                error_count += 1
                console.print(f"  [red]✗ TIMEOUT[/red]  [dim]{short_url(current)}[/dim]")
            except Exception as e:
                error_count += 1
                console.print(f"  [red]✗ ERROR[/red] [dim]{type(e).__name__}: {e}[/dim]")

        progress.update(task, description="[bold green]✔ Selesai!", completed=1, total=1)

    return all_rows, summary_index, total_fetched, error_count

# =========================
# RUN
# =========================
all_rows, summary_index, total_fetched, error_count = crawl_sitemap(SITEMAP_URL)

# =========================
# RINGKASAN AKHIR
# =========================
END_TIME = datetime.now()
DURATION = (END_TIME - START_TIME).total_seconds()

console.print()
console.rule("[bold green]✅ CRAWLING SELESAI[/bold green]")

stats_table = Table(box=box.SIMPLE, show_header=False, padding=(0, 2))
stats_table.add_column(style="dim")
stats_table.add_column(style="bold white")
stats_table.add_row("🗂  Sitemap diproses",    str(total_fetched))
stats_table.add_row("🔗  Total URL diekstrak", f"[bold green]{len(all_rows)}[/bold green]")
stats_table.add_row("❌  Error",               f"[red]{error_count}[/red]" if error_count else "[green]0[/green]")
stats_table.add_row("⏱  Durasi",              f"{DURATION:.1f} detik")
stats_table.add_row("💾  Output file",          f"[cyan]{OUTPUT_FILE}[/cyan]")
console.print(Panel(stats_table, title="[bold]📊 Statistik[/bold]", border_style="green"))

if all_rows:
    df = pd.DataFrame(all_rows)

    # Tabel entry type
    type_counts = df["entry_type"].value_counts()
    type_table  = Table(title="Entry Type", box=box.ROUNDED, border_style="cyan",
                        show_header=True, header_style="bold cyan")
    type_table.add_column("Type",  style="white")
    type_table.add_column("Count", style="bold green", justify="right")
    type_table.add_column("Bar",   style="cyan", no_wrap=True)
    max_count = type_counts.max()
    for etype, cnt in type_counts.items():
        bar_len = int(cnt / max_count * 18)
        bar     = "█" * bar_len + "░" * (18 - bar_len)
        style   = ENTRY_TYPE_STYLE.get(etype, "[white]")
        type_table.add_row(f"{style}{etype}[/]", str(cnt), f"[cyan]{bar}[/cyan]")

    # Tabel per source
    src_table = Table(title="Per Sitemap Source", box=box.ROUNDED, border_style="blue",
                      show_header=True, header_style="bold blue")
    src_table.add_column("Sitemap",   style="dim", max_width=55, no_wrap=True)
    src_table.add_column("URLs",      style="bold green", justify="right")
    per_src = df.groupby("sitemap_source").size().reset_index(name="url_count") \
                .sort_values("url_count", ascending=False)
    for _, row in per_src.iterrows():
        src_table.add_row(short_url(row["sitemap_source"], 55), str(row["url_count"]))

    console.print(Columns([type_table, src_table], equal=False, expand=False))

    # Export & download
    console.print()
    console.rule("[bold cyan]💾 EXPORT[/bold cyan]")
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    console.print(f"\n  [bold green]✔[/bold green] File     : [cyan]{OUTPUT_FILE}[/cyan]")
    console.print(f"  [bold green]✔[/bold green] Rows     : [bold]{len(df)}[/bold] baris")
    console.print(f"  [bold green]✔[/bold green] Encoding : UTF-8 with BOM\n")
    files.download(OUTPUT_FILE)
    console.print(f"  [bold cyan]⬇  Download dimulai...[/bold cyan]\n")

else:
    console.print("\n[bold red]❌ Gagal mendapatkan data atau sitemap kosong.[/bold red]")